In [1]:
%load_ext cudf.pandas
import sys, os

# point this at the directory *containing* tpch/
tpch_parent = os.path.abspath(os.path.join(os.getcwd(), ".."))  
sys.path.insert(0, tpch_parent)

from tpch import load_lineitem, q01
import pandas as pd
DATA_ROOT = "/home/dias-benchmarks/tpch/data/factor_1"
STORAGE_OPTS = {}  # or load from JSON

# # load just the datasets q01 needs:
# lineitem = load_lineitem(DATA_ROOT, **STORAGE_OPTS)


In [2]:
# load just the datasets q01 needs:
lineitem = load_lineitem(DATA_ROOT, **STORAGE_OPTS)

Index(['L_ORDERKEY', 'L_PARTKEY', 'L_SUPPKEY', 'L_LINENUMBER', 'L_QUANTITY',
       'L_EXTENDEDPRICE', 'L_DISCOUNT', 'L_TAX', 'L_RETURNFLAG',
       'L_LINESTATUS', 'L_SHIPDATE', 'L_COMMITDATE', 'L_RECEIPTDATE',
       'L_SHIPINSTRUCT', 'L_SHIPMODE', 'L_COMMENT', 'L_DUMMY'],
      dtype='object')


In [3]:
%%time
_ = q01(lineitem)

  L_RETURNFLAG L_LINESTATUS  L_QUANTITY  L_EXTENDEDPRICE    DISC_PRICE  \
0            A            F    37734107     5.658655e+10  5.375826e+10   
1            N            F      991417     1.487505e+09  1.413082e+09   
2            N            O    74476040     1.117017e+11  1.061182e+11   
3            R            F    37719753     5.656804e+10  5.374129e+10   

         CHARGE    AVG_QTY     AVG_PRICE  L_DISCOUNT  L_ORDERKEY  
0  5.590907e+10  25.522006  38273.129735    0.049985     1478493  
1  1.469649e+09  25.516472  38284.467761    0.050093       38854  
2  1.103670e+11  25.502227  38249.117989    0.049997     2920374  
3  5.588962e+10  25.505794  38250.854626    0.050009     1478870  
Q01 Execution time (s): 0.256310
CPU times: user 123 ms, sys: 48.7 ms, total: 172 ms
Wall time: 256 ms


In [4]:
%%time

def q01_cudf(lineitem):
    # 1) Ensure shipdate is datetime on GPU
    # if not cudf.api.types.is_datetime64_any_dtype(lineitem['L_SHIPDATE']):
    #     lineitem['L_SHIPDATE'] = cudf.to_datetime(lineitem['L_SHIPDATE'])
    
    # 2) Define cutoff and select only the columns we need
    cutoff = pd.Timestamp('1998-09-02')
    cols = [
        'L_QUANTITY','L_EXTENDEDPRICE','L_DISCOUNT','L_TAX',
        'L_RETURNFLAG','L_LINESTATUS','L_SHIPDATE','L_ORDERKEY'
    ]
    df = lineitem[cols]
    
    # 3) Filter on ship date
    df = df[df['L_SHIPDATE'] <= cutoff]
    
    # 4) Turn the two grouping keys into categoricals to accelerate the shuffle
    df['L_RETURNFLAG']  = df['L_RETURNFLAG'].astype('category')
    df['L_LINESTATUS']  = df['L_LINESTATUS'].astype('category')
    
    # 5) Compute discounted price and charge in two GPU kernels
    df['DISC_PRICE'] = df['L_EXTENDEDPRICE'] * (1 - df['L_DISCOUNT'])
    df['CHARGE']     = df['DISC_PRICE'] * (1 + df['L_TAX'])
    
    # 6) Group and aggregate everything in one go
        # 6) Group and compute all sums, means, and counts in one go
    aggs = {
        'L_QUANTITY':      ['sum','mean'],
        'L_EXTENDEDPRICE': ['sum','mean'],
        'DISC_PRICE':      ['sum'],
        'CHARGE':          ['sum'],
        'L_DISCOUNT':      ['mean'],
        'L_ORDERKEY':      ['count'],
    }
    result = df.groupby(
        ['L_RETURNFLAG','L_LINESTATUS'],
        sort=True
    ).agg(aggs)

    # 7) Flatten the MultiIndex columns to ['col_agg', ...]
    result.columns = [f"{col}_{agg}" for col, agg in result.columns]

    # 8) Rename to your final names, then reset index
    result = result.rename(columns={
        'L_QUANTITY_sum':       'L_QUANTITY',
        'L_QUANTITY_mean':      'AVG_QTY',
        'L_EXTENDEDPRICE_sum':  'L_EXTENDEDPRICE',
        'L_EXTENDEDPRICE_mean': 'AVG_PRICE',
        'DISC_PRICE_sum':       'DISC_PRICE',
        'CHARGE_sum':           'CHARGE',
        'L_DISCOUNT_mean':      'L_DISCOUNT',
        'L_ORDERKEY_count':     'L_ORDERKEY',
    }).reset_index()

    return result


q01_cudf(lineitem)

CPU times: user 126 ms, sys: 76 ms, total: 202 ms
Wall time: 254 ms


,L_RETURNFLAG,L_LINESTATUS,L_QUANTITY,AVG_QTY,L_EXTENDEDPRICE,AVG_PRICE,DISC_PRICE,CHARGE,L_DISCOUNT,L_ORDERKEY
0,A,F,37734107,25.522006,5.658655e+10,38273.129735,5.375826e+10,5.590907e+10,0.049985,1478493
1,N,F,991417,25.516472,1.487505e+09,38284.467761,1.413082e+09,1.469649e+09,0.050093,38854
2,N,O,74476040,25.502227,1.117017e+11,38249.117989,1.061182e+11,1.103670e+11,0.049997,2920374
3,R,F,37719753,25.505794,5.656804e+10,38250.854626,5.374129e+10,5.588962e+10,0.050009,1478870
